<a href="https://colab.research.google.com/github/Pranayshukla0610/mastering-lstm-deep-learning/blob/main/English_to_French_Translator_using_LSTM_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense
)

In [2]:
english_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "good morning",
    "thank you",
    "see you later",
    "what is your name",
    "i love machine learning",
    "this is a book",
    "where are you",
    "how old are you",
    "i like coffee",
    "he is my friend",
    "she is a teacher",
    "welcome home",
    "good night",
    "have a nice day",
    "i am hungry",
    "i am happy",
    "nice to meet you"
]

french_sentences = [
    "<start> bonjour <end>",
    "<start> comment allez vous <end>",
    "<start> je vais bien <end>",
    "<start> bonjour <end>",
    "<start> merci <end>",
    "<start> a plus tard <end>",
    "<start> quel est votre nom <end>",
    "<start> j aime l apprentissage automatique <end>",
    "<start> c est un livre <end>",
    "<start> ou etes vous <end>",
    "<start> quel age avez vous <end>",
    "<start> j aime le cafe <end>",
    "<start> il est mon ami <end>",
    "<start> elle est enseignante <end>",
    "<start> bienvenue a la maison <end>",
    "<start> bonne nuit <end>",
    "<start> bonne journee <end>",
    "<start> j ai faim <end>",
    "<start> je suis heureux <end>",
    "<start> ravi de vous rencontrer <end>"
]

In [3]:
eng_tokenizer = Tokenizer(
    oov_token="<OOV>"
)

eng_tokenizer.fit_on_texts(
    english_sentences
)

In [4]:
eng_vocab_size = (
    len(eng_tokenizer.word_index) + 1
)

print(
    "English Vocabulary:",
    eng_vocab_size
)

English Vocabulary: 43


In [5]:
fr_tokenizer = Tokenizer(
    oov_token="<OOV>"
)

fr_tokenizer.fit_on_texts(
    french_sentences
)

In [6]:
fr_vocab_size = (
    len(fr_tokenizer.word_index)+1
)

print(
    "French Vocabulary:",
    fr_vocab_size
)

French Vocabulary: 51


In [7]:
encoder_input_data = (
    eng_tokenizer.texts_to_sequences(
        english_sentences
    )
)

decoder_input_data = (
    fr_tokenizer.texts_to_sequences(
        french_sentences
    )
)

In [8]:
max_eng_len = max(
    len(seq)
    for seq in encoder_input_data
)

max_fr_len = max(
    len(seq)
    for seq in decoder_input_data
)

In [9]:
print(max_eng_len)
print(max_fr_len)

4
7


In [10]:
encoder_input_data = pad_sequences(
    encoder_input_data,
    maxlen = max_eng_len,
    padding = 'post'
)

decoder_input_data = pad_sequences(
    decoder_input_data,
    maxlen = max_fr_len,
    padding = 'post'
)

In [11]:
decoder_target_data = np.zeros_like(
    decoder_input_data
)

decoder_target_data[:,:-1] = (
    decoder_input_data[:,1:]
)

In [12]:
encoder_inputs = Input(
    shape=(max_eng_len,)
)

In [13]:
encoder_embedding = Embedding(
    input_dim = eng_vocab_size,
    output_dim = 32
)(
    encoder_inputs
)

In [14]:
encoder_lstm = LSTM(
    64,
    return_state=True
)

In [15]:
encoder_outputs, state_h, state_c = (
    encoder_lstm(
        encoder_embedding
    )
)
encoder_states = [
    state_h,
    state_c
]

In [16]:
decoder_inputs = Input(
    shape=(max_fr_len,)
)

In [17]:
decoder_embedding = Embedding(
    input_dim = fr_vocab_size,
    output_dim = 32
)(
    decoder_inputs
)

In [18]:
decoder_lstm = LSTM(
    64,
    return_sequences=True,
    return_state=True
)

In [19]:
decoder_outputs, _, _ = (
    decoder_lstm(
        decoder_embedding,
        initial_state=encoder_states
    )
)

In [21]:
decoder_dense = Dense(
    fr_vocab_size,
    activation='softmax'
)

decoder_outputs = (
    decoder_dense(
        decoder_outputs
    )
)

In [23]:
from tensorflow.keras.models import Model

model = Model(
    [encoder_inputs,
    decoder_inputs],

    decoder_outputs
)

In [25]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [26]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4, 32)     │      1,376 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 7, 32)     │      1,632 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 64),      │     24,832 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 7, 64),   │     24,832 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 7, 51)     │      3,315 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 55,987 (218.70 KB)

 Trainable params: 55,987 (218.70 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
history = model.fit(
    [encoder_input_data,
     decoder_input_data],
    np.expand_dims(
        decoder_target_data,
        -1
    ),
    batch_size=4,
    epochs=200,
    verbose=1
)

Epoch 1/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3214 - loss: 3.9188   
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 3.8808 
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4214 - loss: 3.8186
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 3.7011 
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 3.4227 
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 2.8685 
Epoch 7/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4214 - loss: 2.3261
Epoch 8/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 2.3171  
Epoch 9/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 2.2846 
Epoch 10/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 2.1895 
Epoch 11/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - loss: 2.1413 
Epoch 12/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4214 - 

In [32]:
reverse_fr_index = {
    index: word
    for word, index in fr_tokenizer.word_index.items()
}

print(list(reverse_fr_index.items())[:10])

[(1, '<OOV>'), (2, 'start'), (3, 'end'), (4, 'vous'), (5, 'est'), (6, 'j'), (7, 'bonjour'), (8, 'je'), (9, 'a'), (10, 'quel')]


In [33]:
encoder_model = Model(
    encoder_inputs,
    encoder_states
)

In [34]:
decoder_state_input_h = Input(
    shape=(64,)
)

decoder_state_input_c = Input(
    shape=(64,)
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

In [35]:
decoder_outputs, state_h, state_c = (
    decoder_lstm(
        decoder_embedding,
        initial_state=decoder_states_inputs
    )
)

In [36]:
decoder_outputs = decoder_dense(
    decoder_outputs
)

In [37]:
decoder_model = Model(
    [
        decoder_inputs
    ] + decoder_states_inputs,

    [
        decoder_outputs,
        state_h,
        state_c
    ]
)

In [38]:
def translate_sentence(sentence):

    sequence = eng_tokenizer.texts_to_sequences(
        [sentence]
    )

    sequence = pad_sequences(
        sequence,
        maxlen=max_eng_len,
        padding='post'
    )

    states_value = encoder_model.predict(
        sequence,
        verbose=0
    )

    start_token = (
        fr_tokenizer.word_index['start']
    )

    target_seq = np.array(
        [[start_token]]
    )

    decoded_sentence = []

    for _ in range(max_fr_len):

        output_tokens, h, c = (
            decoder_model.predict(
                [target_seq] + states_value,
                verbose=0
            )
        )

        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )

        sampled_word = (
            reverse_fr_index.get(
                sampled_token_index,
                ""
            )
        )

        if (
            sampled_word == "end"
            or sampled_word == ""
        ):
            break

        decoded_sentence.append(
            sampled_word
        )

        target_seq = np.array(
            [[sampled_token_index]]
        )

        states_value = [h, c]

    return " ".join(decoded_sentence)

In [39]:
print(
    translate_sentence(
        "hello"
    )
)

bonjour


In [40]:
model.save(
    "english_french_lstm.h5"
)